In [ ]:
# default_exp training.plsr

# Training & validation (PLSR)

> Various utilities function to train and evaluate the Partial Least Squares Regression baseline model

In [ ]:
#hide
from nbdev.showdoc import *

In [ ]:
#|export
from __future__ import annotations

# Python utils
from collections import OrderedDict
from tqdm.auto import tqdm

# mirzai utils
from mirzai.data.loading import load_kssl
from mirzai.data.selection import (select_y, select_tax_order, select_X)
from mirzai.data.transform import (log_transform_y, SNV, TakeDerivative,
                                   DropSpectralRegions, CO2_REGION)

# Data science stack
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error

from fastcore.test import *
from fastcore.transform import compose

In [ ]:
#|export
def compute_valid_curve(X:np.ndarray, # Spectra with shape (n_samples, n_wavenumbers)
                        y:np.ndarray, # Target with shape (n_samples) 
                        X_names:np.ndarray, # Wavenumbers name with shape (n_wavenumbers)
                        mask:np.ndarray=None, # Mask (e.g for specific Soil Taxonomy Orders) with shape (n_samples)
                        pls_components:list=range(5, 120, 5), # List of PLSR components to try
                        seeds:list=range(20), # List of random seeds to use for multiple train/test splits
                        test_size:float=0.1 # Train/test ratio
                       ):
    "Train the PLSR model on training & evaluate it on the valid set as # pls components increases"
    history = OrderedDict({'pls_components': pls_components, 
                           'train_score': [], 
                           'valid_score': []})
    for seed in tqdm(seeds):
        if mask is None:
            mask = np.full(len(X), True)
        X_train, X_valid, y_train, y_valid = train_test_split(X[mask, : ], y[mask],
                                                              test_size=test_size, 
                                                              random_state=seed)

        train_score = []
        valid_score = []
        for cpts in tqdm(pls_components):
            pipe = Pipeline([('snv', SNV()),
                            ('derivative', TakeDerivative(window_length=11, polyorder=1)), 
                            ('dropper', DropSpectralRegions(X_names, regions=CO2_REGION)),
                            ('model', PLSRegression(n_components=cpts))])

            pipe.fit(X_train, y_train)

            train_score.append(mean_squared_error(pipe.predict(X_train), y_train))
            valid_score.append(mean_squared_error(pipe.predict(X_valid), y_valid))

        history['train_score'].append(train_score)
        history['valid_score'].append(valid_score)
    
    return history

In [ ]:
src_dir = './files'
fnames = ['spectra-features-smp.npy', 'spectra-wavenumbers-smp.npy', 
          'depth-order-smp.npy', 'target-smp.npy', 
          'tax-order-lu-smp.pkl', 'spectra-id-smp.npy']

X, X_names, depth_order, y, tax_lookup, X_id = load_kssl(src_dir, fnames=fnames)
transforms = [select_y, select_tax_order, select_X, log_transform_y]

data = X, y, X_id, depth_order
X, y, X_id, depth_order = compose(*transforms)(data)

In [ ]:
# On all data over different random seeds
compute_valid_curve(X, 
                    y,
                    X_names,
                    mask=None,
                    pls_components=range(5, 20, 5),
                    seeds=range(2),
                    test_size=0.1)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

OrderedDict([('pls_components', range(5, 20, 5)),
             ('train_score',
              [[0.054833092853893345,
                0.01771527542918605,
                0.008755568234449129],
               [0.04787050912971831,
                0.020830987706066652,
                0.010451989937235404]]),
             ('valid_score',
              [[0.09792013659588887, 0.14192822375546943, 0.2040144479474275],
               [0.23643641839954527,
                0.32857102119835635,
                0.25784614375605736]])])

In [ ]:
#|export
class PLS_model():
    "Partial Least Squares model runner"
    def __init__(self, n_components):
        self.n_components = n_components
        self.model = None

    def fit(self, data):
        X, y = data
        self.model = Pipeline([('snv', SNV()),
                         ('derivative', TakeDerivative()), 
                         ('dropper', DropSpectralRegions(X_names, regions=CO2_REGION)),
                         ('model', PLSRegression(n_components=self.n_components))])
        self.model.fit(X, y)
        return self

    def predict(self, data):
        X, y = data
        return self.model.predict(X)

    def eval(self, data, is_log=True):
        X, y = data
        return eval_reg(y, self.model.predict(X))

In [ ]:
#|export
class Evaluator():
    def __init__(self, data, depth_order, 
                 seeds=range(20), n_components=10, split_ratio=0.1):
        self.seeds = seeds
        self.X, self.y = data 
        self.depth_order = depth_order
        self.split_ratio = split_ratio
        self.n_components = n_components

        self.models = []
        self.perfs = OrderedDict({'train': [], 'test': []})

    def train_multiple(self):
        for seed in tqdm(self.seeds):
            X_train, X_test, y_train, y_test, depth_order_train, depth_order_test = self._splitter(seed)
            model = PLS_model(self.n_components)
            model.fit((X_train, y_train))
            self.models.append(model)

    def eval_on_train(self, reducer):
        perfs = []
        for i, seed in enumerate(self.seeds):
            X_train, X_test, y_train, y_test, _, _ = self._splitter(seed)
            perf = self.models[i].eval((X_train, y_train))
            perf['n'] = len(X_train)
            perfs.append(perf)
        if reducer:
            perfs = self.reduce(perfs, reducer)
        return perfs

    def eval_on_test(self, order=-1, reducer=None):
        perfs = []
        for i, seed in enumerate(self.seeds):
            X_train, X_test, y_train, y_test, depth_order_train, depth_order_test = self._splitter(seed)
            if order != - 1:
                mask = depth_order_test[:, 1] == order
                X_test, y_test = X_test[mask, :], y_test[mask]
            perf = self.models[i].eval((X_test, y_test))
            perf['n'] = len(X_test)
            perfs.append(perf)
        if reducer:
            perfs = self.reduce(perfs, reducer)
        return perfs

    def _splitter(self, seed):
        return train_test_split(self.X, self.y, self.depth_order,
                                test_size=self.split_ratio, 
                                random_state=seed)

    def reduce(self, perfs, fn=np.mean):
        results = {}
        for metric in perfs[0].keys():
            result = fn(np.array([perf[metric] for perf in perfs]))
            results[metric] = result        
        return results